# 04 · From the gait comparison to an explicit learning task

[Issue #33](https://github.com/haidmoham/spider/issues/33) · **Continuation of notebook 03. Prepared; no new run.**

Notebook 03 ends with `gait-policy-change-observe`: the fixed shuffle versus your
speed-feedback gait, with sweep/lift diagnostics. This notebook starts from that work.
The recorded five-second comparison moved the fixed shuffle about **+0.1406 m** forward;
the feedback policy moved about **−0.0380 m** forward and **+0.1507 m** sideways.
The sweep reached both bounds. The cause remains open; we do not need to fix that gait before RL.
See [the dated receipt](../notes/2026-09-14-policy-and-shuffle.md) and notebook 03's saved outputs.

**Next question:** What observation, action, and reward contract would let us compare those
behaviors—and later learn from them—without confusing a working update with a useful task?

Your corrected geometry probe, feedback-policy attempt, and REINFORCE update are the starting point.
Do not repeat them as prerequisites. Reuse your prior explanations and revise only what the new
contract needs. Notebook 03 keeps its original code and measured outputs.

**First work package:** start with the existing forward-speed observation, sweep/lift policy,
and 18-offset adapter below. Explain which information the learning policy needs beyond that
feedback input. Propose a minimal reward that distinguishes forward progress from sideways motion.
Then implement only the observation and reward changes needed to inspect the existing behaviors.
Review this before collecting a new rollout.

The later sections turn that proposal into one explicit transition:
`(obs_t, action_t, reward_t, obs_t1, terminated, info)`.
Use the **C-1N pairing** kernel. The carried code is defined here so notebook 03 need not be rerun.
TODO functions remain yours. No trainer or new experiment has run.

## 1. Setup

Imports and model inspection only. This creates the existing adapter and its neutral reset;
it does not advance physics. The adapter remains the only stepping interface.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "spider" / "learning.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import mujoco
import numpy as np
import pandas as pd
from spider.learning import LearningSimulation, record_policy, plot_recordings
from spider.simulation import neutral_targets

sim = LearningSimulation()
actuator_names = [sim.model.actuator(i).name for i in range(sim.model.nu)]
print("Physics timestep (s):", sim.model.opt.timestep)
display(pd.DataFrame({
    "index": range(sim.model.nu), "actuator": actuator_names,
    "neutral_rad": neutral_targets(),
    "target_min_rad": sim.model.actuator_ctrlrange[:, 0],
    "target_max_rad": sim.model.actuator_ctrlrange[:, 1],
}))

### Carry forward notebook 03's code

The next cells copy the existing geometry helper, fixed shuffle, and feedback policy.
These are references, not new solutions. Source: notebook 03 cells `gait-joint-direction-probe`,
`4105067d`, `d5f5f032`, and `basic-gait-parameter-policy` at commit `e6ba912`.

The geometry setup uses forward kinematics at time zero. It does not integrate physics.
The policy definitions do not run a rollout. Existing settings are retained: sweep 0.04 rad,
lift 0.05 rad, period 0.8 s, target speed 0.03 m/s, and feedback gain 0.5.
The previous comparison used 10 physics steps per action and 250 actions (5 s).
These are reference settings, not silently accepted MDP choices.

`gait_parameter_policy` reads `state.torso_velocity[0]` and emits sweep/lift.
`gait_from_parameters` expands those into 18 radian offsets. Keep those two interfaces distinct:
issue #33 retains the 18-offset action interface. Any normalized policy action needs an explicit mapping.
The existing reference gait also uses time/phase; consider what omitting phase means for a learned policy.

In [ ]:
# Next: which joint moves each foot forward/backward near the neutral pose?
# Runs geometry diagnostics only: no time stepping, viewer, training, or gait search.
# During stance, the floor supports the body against gravity. A backward push
# on the floor can produce a forward reaction on the body. First check the geometry.
from spider.simulation import FOOT_NAMES

probe_sim = LearningSimulation()  # Separate scratch state; preserves the policy runs.
PROBE_DELTA_RAD = 0.01
PROBE_JOINTS = ("coxa", "hip", "knee")


def foot_at_offset(leg, joint, offset_rad):
    """Geometry helper: one joint offset from neutral -> foot XYZ in torso frame (m)."""
    probe_sim.reset()
    model, data = probe_sim.model, probe_sim.data
    joint_id = model.joint(f"{leg}_{joint}").id
    address = model.jnt_qposadr[joint_id]
    data.qpos[address] += offset_rad  # Set an angle directly; this is not an actuator command.
    mujoco.mj_forward(model, data)  # Recompute geometry without advancing time.
    torso_id = model.body("torso").id
    foot_id = model.geom(f"{leg}_foot").id
    rotation = data.xmat[torso_id].reshape(3, 3)
    return (rotation.T @ (data.geom_xpos[foot_id] - data.xpos[torso_id])).copy()


def foot_motion_per_radian(leg, joint, delta=PROBE_DELTA_RAD):
    """Return a 3-vector: local foot motion [dx/dq, dy/dq, dz/dq], in m/rad."""
    # Two-sided finite difference; retain the diagnostics when changing this probe.
    plus = foot_at_offset(leg, joint, delta)
    minus = foot_at_offset(leg, joint, -delta)
    return (plus - minus) / (2 * delta)

In [ ]:
# Geometry-informed tripod gait.
# This is a hypothesis / visualization, not an earned locomotion result.

import numpy as np

LEGS = (
    "front_left",
    "front_right",
    "middle_left",
    "middle_right",
    "rear_left",
    "rear_right",
)

TRIPOD_A = {"front_left", "middle_right", "rear_left"}
TRIPOD_B = set(LEGS) - TRIPOD_A

# Small offsets. Keep this boring initially.
COXA_SWEEP_RAD = 0.04
LIFT_RAD = 0.05
GAIT_PERIOD_S = 0.8


# Learn local directions from geometry instead of guessing signs.
geometry = {}

for leg in LEGS:
    coxa_j = foot_motion_per_radian(leg, "coxa")

    hip_j = foot_motion_per_radian(leg, "hip")
    knee_j = foot_motion_per_radian(leg, "knee")

    lift_joint, lift_j = max(
        [("hip", hip_j), ("knee", knee_j)],
        key=lambda item: abs(item[1][2]),
    )

    # sign that makes the foot move backward in torso +X coordinates
    backward_sign = -np.sign(coxa_j[0])

    # sign that makes the selected joint move the foot upward (+Z)
    lift_sign = np.sign(lift_j[2])

    geometry[leg] = {
        "backward_sign": backward_sign,
        "lift_joint": lift_joint,
        "lift_sign": lift_sign,
        "dx_dq": coxa_j[0],
        "dz_dq": lift_j[2],
    }

geometry

In [ ]:
def geometry_gait(state):
    offsets = np.zeros(18)

    phase = 2 * np.pi * state.time / GAIT_PERIOD_S

    for leg_index, leg in enumerate(LEGS):
        # Opposite phase for alternating tripods.
        leg_phase = phase if leg in TRIPOD_A else phase + np.pi

        swing = np.sin(leg_phase) > 0

        info = geometry[leg]

        coxa_index = leg_index * 3
        hip_index = coxa_index + 1
        knee_index = coxa_index + 2

        # stance: push foot backward
        # swing: bring foot forward
        direction = -1 if swing else 1

        offsets[coxa_index] = (
            direction
            * info["backward_sign"]
            * COXA_SWEEP_RAD
        )

        # Lift only during swing.
        if swing:
            lift_index = hip_index if info["lift_joint"] == "hip" else knee_index
            offsets[lift_index] = info["lift_sign"] * LIFT_RAD

    return offsets



# Basic policy over gait parameters: measured state -> sweep/lift -> 18 joint offsets.
# The fixed geometry_gait above remains the control. No training or rollout runs here.
# Period stays fixed: changing time / period on every call would jump the gait phase.
TARGET_SPEED_M_S = 0.03  # Editable first target, not a demonstrated capability.


SPEED_GAIN = 0.5  # rad / (m/s); a starting hypothesis, not a tuned gain.
SWEEP_MIN_RAD, SWEEP_MAX_RAD = 0.0, 0.08


def gait_parameter_policy(state, target_speed_m_s=TARGET_SPEED_M_S):
    measured_speed = state.torso_velocity[0]
    speed_error = target_speed_m_s - measured_speed

    raw_sweep_rad = COXA_SWEEP_RAD + SPEED_GAIN * speed_error
    sweep_rad = np.clip(raw_sweep_rad, SWEEP_MIN_RAD, SWEEP_MAX_RAD)

    return float(sweep_rad), float(LIFT_RAD)


# Existing gait plumbing, copied with explicit parameters instead of mutable globals.
def gait_from_parameters(state, sweep_rad, lift_rad):
    offsets = np.zeros(18)

    phase = 2 * np.pi * state.time / GAIT_PERIOD_S

    for leg_index, leg in enumerate(LEGS):
        # Opposite phase for alternating tripods.
        leg_phase = phase if leg in TRIPOD_A else phase + np.pi

        swing = np.sin(leg_phase) > 0

        info = geometry[leg]

        coxa_index = leg_index * 3
        hip_index = coxa_index + 1
        knee_index = coxa_index + 2

        # stance: push foot backward
        # swing: bring foot forward
        direction = -1 if swing else 1

        offsets[coxa_index] = (
            direction
            * info["backward_sign"]
            * sweep_rad
        )

        # Lift only during swing.
        if swing:
            lift_index = hip_index if info["lift_joint"] == "hip" else knee_index
            offsets[lift_index] = info["lift_sign"] * lift_rad

    return offsets


def basic_gait_policy(state):
    sweep_rad, lift_rad = gait_parameter_policy(state)
    return gait_from_parameters(state, sweep_rad, lift_rad)

### Next: fix this draft reward

Run the cell below on its own. Edit `draft_reward`, then rerun it and inspect `draft_log`.
The inputs are synthetic motions, so this cell needs no earlier kernel variables and runs no physics.
The table keeps raw motion, reward terms, and total score side by side.

Aim for a reward you can defend for the gait comparison from notebook 03.
This is an intentionally rough draft, not an accepted task definition.

In [ ]:
# Draft task reward. Edit draft_reward; the table shows what it actually scores.
# These are synthetic transitions, not simulator runs or measured robot evidence.
import numpy as np
import pandas as pd
from types import SimpleNamespace
from IPython.display import display

DRAFT_PROGRESS_WEIGHT = 1.0
DRAFT_HEIGHT_WEIGHT = 0.1


def draft_reward(before, after):
    progress = float(np.linalg.norm(after.torso_velocity[:2]))
    height = float(after.torso_position[2])
    terms = {
        "progress_term": DRAFT_PROGRESS_WEIGHT * progress,
        "height_term": DRAFT_HEIGHT_WEIGHT * height,
    }
    return sum(terms.values()), terms


# Transparent inputs: label, starting XYZ (m), ending XYZ (m), elapsed time (s).
draft_cases = [
    ("still", (0, 0, 0.3), (0, 0, 0.3), 0.02),
    ("forward", (0, 0, 0.3), (0.0006, 0, 0.3), 0.02),
    ("backward", (0, 0, 0.3), (-0.0006, 0, 0.3), 0.02),
    ("sideways", (0, 0, 0.3), (0, 0.0006, 0.3), 0.02),
    ("lower forward", (0, 0, 0.1), (0.0006, 0, 0.1), 0.02),
    ("longer action", (0, 0, 0.3), (0.0012, 0, 0.3), 0.04),
]
draft_rows = []
for label, start_xyz, end_xyz, dt_s in draft_cases:
    start_xyz, end_xyz = np.asarray(start_xyz), np.asarray(end_xyz)
    velocity = (end_xyz - start_xyz) / dt_s
    before = SimpleNamespace(time=0.0, torso_position=start_xyz.copy(),
                             torso_velocity=velocity.copy())
    after = SimpleNamespace(time=dt_s, torso_position=end_xyz.copy(),
                            torso_velocity=velocity.copy())
    reward, terms = draft_reward(before, after)
    draft_rows.append({
        "case": label, "dt_s": dt_s,
        "x_before_m": start_xyz[0], "x_after_m": end_xyz[0],
        "dx_m": end_xyz[0] - start_xyz[0], "dy_m": end_xyz[1] - start_xyz[1],
        "vx_m_s": velocity[0], "vy_m_s": velocity[1], "z_after_m": end_xyz[2],
        **terms, "reward": reward,
    })

draft_log = pd.DataFrame(draft_rows).set_index("case")
print("SYNTHETIC INPUTS | world XYZ in metres | velocities in m/s")
print("Weights:", {"progress": DRAFT_PROGRESS_WEIGHT, "height": DRAFT_HEIGHT_WEIGHT})
with pd.option_context("display.max_columns", None, "display.float_format", "{:.6f}".format):
    display(draft_log)
# draft_log retains every row and term for your own inspection.


## 2. Propose the task contract

An MDP describes the state, actions, transitions, rewards, and episode rules.
Your observation is the information the policy receives. It may omit information in the simulator.

Carry over your existing choices and explanations from notebook 03. Fill only the gaps
needed for the RL contract. A prior conversational attempt counts; this is not a restart.

| Choice | Your proposal and reason |
| --- | --- |
| Forward direction and reference frame | Existing comparison uses world +X displacement; explain whether to retain it |
| Small observation and information it omits | TODO |
| Policy action bounds and physical offset mapping | TODO |
| Physics steps per action and elapsed seconds | Existing comparison: 10 steps, 0.02 s; retain or revise with a reason |
| Minimal reward and intended behavior of each term | TODO |
| Horizon and gamma | TODO |
| Termination, truncation, and return handling at each | TODO |

Predict: what should zero action do, and what should one positive then negative joint probe change?
Treat actuator limits as command limits, not proof that a motion is safe.

In [ ]:
# Fill these after discussing the proposal. None means undecided, not a default.
PHYSICS_STEPS = None
HORIZON_ACTIONS = None
GAMMA = None
ACTION_LOW = None
ACTION_HIGH = None
OFFSET_SCALE_RAD = None  # If using normalized actions; document scalar vs per-joint scale.
SEEDS = None             # Fixed comparison seeds, chosen before runs.

# Once PHYSICS_STEPS is chosen:
# print("Policy action duration (s):", PHYSICS_STEPS * sim.model.opt.timestep)

## 3. Make the observation inspectable

Read `MeasuredState` and `measured_state` in [simulation.py](../../spider/simulation.py).
Candidate ingredients include torso motion, orientation, joint angles/velocities, and contacts.
Start from the feedback policy's existing forward-velocity input. Explain what to retain or add. Check how each source field is computed before assigning its frame or ordering.
Do not add normalization until measured scales justify it.

Fill one row per component. For orientation, name the representation and element order.
For contacts, define a fixed foot order. Document any transform or finite difference.

| Output slice | Source / transform | Shape | Units | Frame / order | Why included |
| --- | --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO | TODO |

Implement `observe`. Return a finite, one-dimensional NumPy array with a stable order.

In [ ]:
def observe(state):
    """MeasuredState -> observation vector in the documented order."""
    raise NotImplementedError("Choose and implement the observation.")

In [ ]:
# Run after implementing observe. Reset only; no rollout.
# first = observe(sim.reset())
# second = observe(sim.reset())
# assert first.ndim == 1 and np.isfinite(first).all()
# np.testing.assert_array_equal(first, second)
# display(pd.DataFrame({"index": range(first.size), "reset_value": first}))
# Compare values with your units/frame table. Add named plots after the signed probes.

## 4. Define action scaling and clipping

`LearningSimulation.step(offsets, physics_steps=...)` accepts **18 finite radian offsets**
in actuator order. It adds `neutral_targets()` and clips absolute targets to actuator ranges.

Implement the policy-facing mapping. Decide whether an out-of-range policy action is rejected
or clipped. Preserve the requested policy action, mapped offsets, requested targets, applied targets,
and clipping flags. State which action is stored for learning and why.

**Probe plan:** reset before each treatment; compare zero, one bounded positive component,
and its negative. Predict the target mapping separately from the physical joint response.
Test zero mapping against neutral targets. Show clipping explicitly, including a mapping-only
out-of-range check. A clipping check need not advance physics.

In [ ]:
def action_to_offsets(action):
    """Return (offsets_rad, info) for one policy action of shape (18,).

    info must preserve policy input, scaling, and any policy-space clipping.
    The transition adds requested/applied physical targets and their clipping flags.
    """
    raise NotImplementedError("Define bounds, scaling, and stored-action semantics.")

## 5. Propose the reward before scoring runs

The earlier `next_x + 0.1 * next_z` isolated an optimizer update. It is not an established task reward.
Propose a minimal forward-progress term. Consider displacement per action versus velocity;
explain how changing the action duration would affect the score. Add shaping only for a named reason.

| Term | Formula and units | Weight | Intended effect | Possible failure |
| --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO |

Predict the ranking of **neutral**, **fixed geometry-informed tripod shuffle**, and
**bounded random** behavior under your reward. Explain what observation could reject your proposal.

In [ ]:
def reward_terms(before, action, after, info):
    """Return a dict of named, weighted scalar contributions; reward is their sum."""
    raise NotImplementedError("Implement your proposed task reward.")

## 6. State the episode boundaries

Termination means the task reaches an ending condition. Truncation means collection stops at a limit.
Name any fall, invalid-state, or simulator-reset criteria. State whether a time limit is part of the task
or only a collection limit. Record both flags, including when they coincide.

For each boundary, specify what happens to future returns and any eventual bootstrap value.
Keep gamma explicit. Do not add a critic or GAE here.

In [ ]:
def episode_boundary(before, after, action_index, info):
    """Return (terminated, truncated, reason); action_index is zero-based."""
    raise NotImplementedError("Implement the agreed episode rules.")

## 7. Implement one transition

Use the interfaces above. Keep physics inside `LearningSimulation.step`.

1. Build `obs_t` from the current measurement. Copy the chosen learning action.
2. Map the action to offsets. Preserve the requested targets before stepping.
3. Call the adapter once with the chosen physics-step count.
4. Build `obs_t1`. Read applied targets from `simulation.data.ctrl.copy()`.
5. Record elapsed time and both policy-space and target-space clipping.
6. Calculate and log each reward term. Apply the episode rules.
7. Return the transition below and the next raw measurement.

Use `info` for `reward_terms`, `truncated`, boundary reason, times, and action mapping details.
Keep raw measurements available for reward calculations without silently adding them to policy input.

In [ ]:
def transition(simulation, before, action, action_index):
    """Return (sample, after).

    sample = (obs_t, action_t, reward_t, obs_t1, terminated, info)
    after is the next MeasuredState used by the following call.
    """
    raise NotImplementedError("Compose the agreed contract using LearningSimulation.step.")

## 8. Compare three fixed behaviors

**Execution checkpoint:** run only after you have reviewed the proposed semantics and implementation.
Preparing this notebook does not authorize these runs.

Use neutral, the existing `geometry_gait` fixed shuffle in notebook 03, and a bounded random policy.
Use the carried `geometry_gait` above with its existing parameters. Confirm how its radian
offsets enter the new policy-action interface. Keep `basic_gait_policy` as an additional failure
reference if useful; it does not replace the required random comparison. Notebook 03 need not run.

Use the same reset, timing, horizon, and reward for all three. Declare seeds before random sampling.
Log total and per-term reward, duration, world displacement, and boundary reason. Account for early
termination when comparing totals. Plot the term totals and displacement beside the table.

Capture exact-state replays. Before interpreting a ranking, view the measured treatments and neutral
control with labels and changed parameters visible. Label any reconstruction as a rerun.

**Your interpretation:** TODO. Does the ranking match your prediction? What failure would change the reward?

In [ ]:
# Implement the bounded collector after reviewing sections 2–7.
# Use transition(...) for each action. Stop at termination or truncation.
# Keep the fixed shuffle, neutral, and seeded random policies separate.
# Record states for replay with spider.recording.TreatmentReplay.
# No rollout or training runs in this scaffold.

## 9. Save and inspect one tiny dataset

Save under `telemetry/issue-33-mdp/<run-label>/` after approving the contract.
Keep this notebook's outputs as the readable run summary.

Include `obs_t`, requested policy action, learning action, mapped offsets, requested/applied targets,
clipping flags, named reward terms, total reward, `obs_t1`, terminated, truncated, and timestamps.
Save the source commit, model hash, full contract, seed, reset conditions, and treatment parameters.

Check before accepting the artifact:

- All transition arrays have the same row count and declared shapes.
- Observations, actions, and rewards are finite.
- Each next timestamp minus its start equals the declared action duration; a reset is a failure.
- `obs_t1[i]` equals `obs_t[i+1]` within an episode.
- Reward equals the sum of its logged terms.
- Episode boundaries stop collection and have explicit reasons.
- Requested versus applied targets explain every clipping flag.

Inspect the saved artifact after loading it back. Keep the measured replay beside it.

In [ ]:
# Add collection, alignment assertions, and artifact save/load here
# after the transition contract is reviewed. Keep each operation in a small cell.

## Review and stop condition

Notebook preparation is complete. Issue #33 remains open until its behavioral evidence is earned.

- [ ] Explain and document the full observation/action/reward/episode contract.
- [ ] Inspect deterministic finite reset observations and zero/signed action probes.
- [ ] Show action scaling and clipping in recorded information.
- [ ] Compare and interpret neutral, fixed shuffle, and bounded random reward terms.
- [ ] Save and reload one short, temporally aligned rollout with explicit boundaries.

Record **prediction → observation → model update → next question** beside the relevant result.
Stop to revise the contract if the probes or reward ranking contradict the intended task.
PPO, GAE, a critic, and locomotion claims remain outside this notebook's scope.